#  Step 2: Fine-Tuning Gemma 2

This notebook fine-tunes **Gemma 2 (2B)** in native **bfloat16**.
Use `pipeline.py finetune` for the canonical end-to-end workflow.
This avoids `bitsandbytes` issues on Windows while still fitting comfortably in 8GB+ VRAM.

In [ ]:
# Install dependencies (No bitsandbytes needed)
%pip install -qU transformers accelerate peft trl datasets

In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

# --- CONFIG ---
MODEL_ID = "google/gemma-2-2b-it" # 2B parameters (~4GB in fp16)
DATA_FILE = "data/rich_commentary_train.jsonl"
OUTPUT_DIR = "models/gemma-2-cricket"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f" Config: Model={MODEL_ID}, Data={DATA_FILE}")

⚙️ Config: Model=google/gemma-2-2b-it, Data=data/rich_commentary_train.jsonl


In [ ]:
# 1. Load Dataset
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f" Could not find {DATA_FILE}. Did you run Notebook 01?")

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = dataset.train_test_split(test_size=0.1) 

print(" Dataset loaded:", dataset)

def format_prompts(example):
    text = f"<start_of_turn>user\n{example['instruction']}\nInput:\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    return {"text": text}

dataset = dataset.map(format_prompts)

📊 Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 90
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10
    })
})


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [7]:
# 2. Load Model (bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16, # Native Half Precision
)
tokenizer.padding_side = 'right'

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
# 3. Training Args (SFTConfig)
from trl import SFTConfig # Re-import to ensure it's defined if cell run out of order

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer, # UPDATED: Renamed from tokenizer in recent TRL
    args=args
)

print("🚂 Starting Training...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


🚂 Starting Training...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.617100,0.922166,0.850766,9662.000000,0.803869
2,0.781400,0.787285,0.681174,19324.000000,0.823098
3,0.589200,0.760450,0.623596,28986.000000,0.817634


TrainOutput(global_step=36, training_loss=0.9157954057057699, metrics={'train_runtime': 76.5059, 'train_samples_per_second': 3.529, 'train_steps_per_second': 0.471, 'total_flos': 410464077410304.0, 'train_loss': 0.9157954057057699, 'epoch': 3.0})

In [ ]:
# 4. Save Model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Model saved to {OUTPUT_DIR}")

✅ Model saved to models/gemma-2-cricket


In [18]:
# 5. Inference Test
prompt = "<start_of_turn>user\nWrite exciting cricket commentary for this ball.\nInput:\n{\"batter\": \"Kohli\", \"bowler\": \"Anderson\", \"runs\": 6}<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)
    
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Write exciting cricket commentary for this ball.
Input:
{"batter": "Kohli", "bowler": "Anderson", "runs": 6}
model
A full toss, a bit too short, and Kohli swings it with a flourish, sending the ball sailing over the ropes for a six!
